In [12]:
# GENETIC ALGORITHM for TSP
# Visit all cities once
# Return to start
# Minimize distance

# Use:
# Tournament Selection
# Ordered Crossover (OX)
# Swap Mutation

import math
import random

cities = [(2,3),(5,7),(1,9),(8,2),(4,5)]

# distance
def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)  # FIXED

# total distance of route
def totaldist(route):
    d = 0
    for i in range(len(route)-1):
        d += distance(cities[route[i]], cities[route[i+1]])
    
    d += distance(cities[route[-1]], cities[route[0]])  # return to start
    return d

# fitness
def fitness(route):
    return 1 / totaldist(route)

# create population
def createpop(size):
    return [random.sample(range(len(cities)), len(cities)) for _ in range(size)]

# tournament selection
def tournament_selection(pop):
    selected = random.sample(pop, 2)   # FIXED
    return max(selected, key=fitness)

# order crossover (OX)
def order_crossover(p1, p2):
    n = len(p1)
    s, e = sorted(random.sample(range(n), 2))  # FIXED

    child = [-1] * n
    child[s:e] = p1[s:e]

    ptr = 0
    for gene in p2:
        if gene not in child:
            while child[ptr] != -1:   # FIXED (chld → child)
                ptr += 1
            child[ptr] = gene

    return child

# swap mutation
def mutation(route):
    i, j = random.sample(range(len(route)), 2)
    route[i], route[j] = route[j], route[i]
    return route

# main GA
def GA():
    pop = createpop(6)

    for gen in range(50):
        pop = sorted(pop, key=fitness, reverse=True)

        best = pop[0]
        print("Gen", gen, "Best Distance:", totaldist(best))  # FIXED

        newpop = []

        while len(newpop) < len(pop):
            # selection
            p1 = tournament_selection(pop)
            p2 = tournament_selection(pop)

            # crossover
            child = order_crossover(p1, p2)

            # mutation
            if random.random() < 0.1:
                child = mutation(child)   # FIXED INDENT

            newpop.append(child)

        pop = newpop   # VERY IMPORTANT (missing before)

    return min(pop, key=totaldist)  # FIXED INDENT

# run
best = GA()

print("Best route:", best)
print("Total distance:", totaldist(best))

Gen 0 Best Distance: 25.518888524155443
Gen 1 Best Distance: 23.873728993095806
Gen 2 Best Distance: 23.873728993095806
Gen 3 Best Distance: 23.873728993095806
Gen 4 Best Distance: 23.873728993095806
Gen 5 Best Distance: 23.873728993095806
Gen 6 Best Distance: 23.873728993095806
Gen 7 Best Distance: 23.873728993095806
Gen 8 Best Distance: 23.873728993095806
Gen 9 Best Distance: 23.873728993095806
Gen 10 Best Distance: 23.873728993095806
Gen 11 Best Distance: 23.873728993095806
Gen 12 Best Distance: 23.873728993095806
Gen 13 Best Distance: 23.873728993095806
Gen 14 Best Distance: 23.873728993095806
Gen 15 Best Distance: 23.873728993095806
Gen 16 Best Distance: 23.873728993095806
Gen 17 Best Distance: 23.873728993095806
Gen 18 Best Distance: 23.873728993095806
Gen 19 Best Distance: 23.873728993095806
Gen 20 Best Distance: 23.873728993095806
Gen 21 Best Distance: 23.873728993095806
Gen 22 Best Distance: 23.873728993095806
Gen 23 Best Distance: 23.873728993095806
Gen 24 Best Distance: 23.8

In [2]:
from collections import deque

hospital = {
    "WardA": {"status": "Critical", "adjacent": ["WardB", "WardC"]},
    "WardB": {"status": "Normal", "adjacent": ["WardA", "WardD"]},
    "WardC": {"status": "Critical", "adjacent": ["WardA", "WardD"]},
    "WardD": {"status": "Empty", "adjacent": ["WardB", "WardC"]}
}

class Agent:
    def __init__(self, start):
        self.current = start
        self.moves = 0
        self.utility = 0
        self.treated = 0

    def bfs(self, start):
        queue = deque([(start, [start])])   # FIXED path start
        visited = set([start])              # FIXED visited

        while queue:
            node, path = queue.popleft()

            if hospital[node]["status"] == "Critical":
                return path

            for neigh in hospital[node]["adjacent"]:
                if neigh not in visited:
                    visited.add(neigh)
                    queue.append((neigh, path + [neigh]))

        return []

    def run(self):
        while any(hospital[w]["status"] == "Critical" for w in hospital):

            # treat current ward if critical
            if hospital[self.current]["status"] == "Critical":
                print(f"Treated at {self.current}")
                hospital[self.current]["status"] = "Treated"
                self.utility += 10
                self.treated += 1
                continue   # 🔥 IMPORTANT FIX

            # find nearest critical ward
            path = self.bfs(self.current)

            if len(path) > 1:
                self.current = path[1]
                self.moves += 1
                print(f"Moved to {self.current}")

        self.utility -= self.moves
        print("Final Utility:", self.utility)

# run
agent = Agent("WardB")
agent.run()

Moved to WardA
Treated at WardA
Moved to WardC
Treated at WardC
Final Utility: 18


In [ ]:
import heapq

grid = [
    ['S', '.', '.', 'W', '.'],
    ['.', '#', '.', '#', '.'],
    ['.', '#', '.', '.', '.'],
    ['.', '.', 'F', '#', 'G']
]

rows, cols = len(grid), len(grid[0])

def cost(cell):
    if cell == 'W': return 3
    if cell == 'F': return 5
    return 1

def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(start, goal):
    pq = [(0, 0, start, [start])]
    visited = set()

    while pq:
        f, g, (x, y), path = heapq.heappop(pq)

        if (x, y) == goal:
            return path, g

        if (x, y) in visited:
            continue

        visited.add((x, y))

        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x+dx, y+dy

            if 0 <= nx < rows and 0 <= ny < cols:
                if grid[nx][ny] != '#':
                    g_new = g + cost(grid[nx][ny])
                    h = heuristic((nx, ny), goal)
                    heapq.heappush(pq, (g_new + h, g_new, (nx, ny), path + [(nx, ny)]))

start = (0,0)
goal = (3,4)

path, cost_val = astar(start, goal)
print("Path:", path)
print("Cost:", cost_val)